# Train YOLOv8 — Waste Sorting Vision (Colab GPU)

Hướng dẫn nhanh:

1. Tải `waste_v5.zip` (2,1 GB) lên **Google Drive**, đặt ở thư mục gốc My Drive.
2. Trên Colab: menu **Runtime → Change runtime type → chọn A100 (hoặc T4)**.
3. Chạy lần lượt các cell 1 → 5 (hoặc **Runtime → Run all**).
4. Nếu Colab bị ngắt giữa chừng (mất kết nối, lỗi `No module named 'ultralytics'`): chạy lại **từ cell 1** — cell 5 tự phát hiện `last.pt` trên Drive và train tiếp từ epoch đang dở.
5. Khi train xong, cell 6 và 7 hiển thị kết quả và tải `best.pt` xuống máy.

Dataset `waste_v5` (đã kiểm tra nhãn): 9 lớp (Paper, Paper Cup, Vinyl, Plastic, Glass, Can, PET, Styrofoam, Battery), 13.038 ảnh train (gồm ảnh Glass/Styrofoam lặp 2 lần) / 3.484 ảnh val. Đã chạy thử trên máy: 0 ảnh hỏng.

Cấu hình train (sửa ở đầu cell 4): `yolov8m`, `imgsz=640`, `batch=32`, tối đa `150` epoch, `patience=30`.
Ước tính trên A100: khoảng 3–4 giờ (chưa đo). Nếu muốn nhanh/rẻ hơn: `MODEL = 'yolov8s.pt'` (khoảng 2 giờ, kém chính xác hơn một chút).
Trên T4 dùng `BATCH = 16` (chậm hơn nhiều, có thể cần nhiều phiên — notebook tự train tiếp).
Kết quả được lưu thẳng vào Drive (`MyDrive/wsv_runs/<NAME>/`) nên không mất khi Colab ngắt kết nối.

Các lần train trước: `waste_v2_yolov8n`, `waste_v3_yolov8n_640`.

In [ ]:
# 1. Kiểm tra GPU (phải thấy Tesla T4 hoặc tương đương)
!nvidia-smi

In [ ]:
# 2. Cài Ultralytics
%pip install -q ultralytics

In [ ]:
# 3. Mount Google Drive và giải nén dataset
from google.colab import drive
drive.mount('/content/drive')

import os
ZIP_PATH = '/content/drive/MyDrive/waste_v5.zip'  # sửa nếu bạn đặt file ở chỗ khác
assert os.path.exists(ZIP_PATH), f'Không tìm thấy {ZIP_PATH} — hãy upload waste_v5.zip lên Drive trước'

!unzip -q -o "$ZIP_PATH" -d /content/
!ls /content/waste_v5
!echo "train: $(ls /content/waste_v5/images/train | wc -l) ảnh, val: $(ls /content/waste_v5/images/val | wc -l) ảnh"

In [ ]:
# 4. Kiểm tra data.yaml và đặt thư mục lưu kết quả trên Drive
import yaml

DATA_YAML = '/content/waste_v5/data.yaml'
with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

data['path'] = '/content/waste_v5'
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
print(open(DATA_YAML).read())

# Cấu hình train — đổi ở đây. Mỗi cấu hình khác nhau được lưu vào thư mục riêng.
MODEL = 'yolov8m.pt'   # yolov8n / yolov8s / yolov8m (chính xác nhất trong 3, chậm nhất)
IMGSZ = 640            # giữ 640: ảnh TACO chỉ có bản 640px nên 1024 không thêm chi tiết
BATCH = 32             # A100: 32; T4: 16; giảm nếu gặp lỗi CUDA out of memory
EPOCHS = 150
PATIENCE = 30

PROJECT = '/content/drive/MyDrive/wsv_runs'
NAME = f"waste_v5_{MODEL.removesuffix('.pt')}_{IMGSZ}"
RUN_DIR = f'{PROJECT}/{NAME}'
print('Thư mục kết quả:', RUN_DIR)

In [ ]:
# 5. Train. Nếu đã có last.pt trên Drive (Colab bị ngắt lần trước) thì tự train tiếp từ epoch đang dở.
import os
from ultralytics import YOLO

LAST = f'{RUN_DIR}/weights/last.pt'
if os.path.exists(LAST):
    print(f'Tìm thấy {LAST} — train tiếp từ checkpoint cuối')
    model = YOLO(LAST)
    results = model.train(resume=True)
else:
    model = YOLO(MODEL)
    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,
        patience=PATIENCE,
        project=PROJECT,
        name=NAME,
        exist_ok=True,
    )

In [ ]:
# 6. Xem kết quả: mAP từng lớp, biểu đồ training và confusion matrix
from IPython.display import Image, display
from ultralytics import YOLO

metrics = YOLO(f'{RUN_DIR}/weights/best.pt').val(data=DATA_YAML, imgsz=IMGSZ, device=0)
for i, class_id in enumerate(metrics.box.ap_class_index):
    name = metrics.names[int(class_id)]
    print(f'{name:<12} mAP50={metrics.box.ap50[i]:.3f}  mAP50-95={metrics.box.ap[i]:.3f}')
print(f'{"ALL":<12} mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}')

display(Image(f'{RUN_DIR}/results.png', width=900))
display(Image(f'{RUN_DIR}/confusion_matrix_normalized.png', width=700))

In [ ]:
# 7. Tải best.pt xuống máy (bản gốc vẫn nằm trên Drive tại RUN_DIR/weights/)
from google.colab import files

print('Checkpoint trên Drive:', f'{RUN_DIR}/weights/best.pt')
files.download(f'{RUN_DIR}/weights/best.pt')